In [ ]:
import ee
import geemap
from utils.variables import PROJECT

import pandas as pd
import geopandas as gpd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Select PA
PA_ID = "11116292"

# Import Christian's grids
grid_gdf = gpd.read_parquet("../data/test_sites_TPA_PSM_GRID.parquet").to_crs(epsg=4326)

# Filter to grids for the selected PA
grid_gdf = grid_gdf[grid_gdf["WDPA_PID"] == PA_ID].copy()

# Convert to ee.FeatureCollection
grid_fc = geemap.geopandas_to_ee(grid_gdf)

# Add a unique ID to each grid cell
cell_IDs = ee.List.sequence(0, grid_fc.size().getInfo() - 1)
featureList = grid_fc.toList(grid_fc.size())
grid_fc = ee.FeatureCollection(
    cell_IDs.map(
        lambda cell_ID: ee.Feature(featureList.get(cell_ID)).set(
            {"cell_ID": cell_ID, "label": None}
        )
    )
)

In [ ]:
# Import covariate datasets (start with just elevation and slope, and then add more)

# Import and process DEM
elevation = (
    ee.ImageCollection("COPERNICUS/DEM/GLO30").filterBounds(grid_fc).select("DEM")
)

slope = elevation.map(lambda tile: ee.Terrain.slope(tile))

elevation = elevation.mosaic().rename("elevation")
slope = slope.mosaic()

# Stack covariates in a single multi-band image
covariates = elevation.addBands(slope)

In [ ]:
# Sample covariate datasets for a random sample of treatment and control cells
# Set a minimum distance of 3 km between samples to avoid spatial autocorrelation
# Ratio of 4 control cells : 1 treatment cell
# Stratify by biome, region, etc.
# switch to stratifiedSample() in the future

n_treatment = 9
n_control = n_treatment * 4

treatment_cells = grid_fc.filter(ee.Filter.eq("protected", 1))
control_cells = grid_fc.filter(ee.Filter.eq("protected", 0))

treatment_sample = (
    treatment_cells.randomColumn(columnName="random", seed=42)
    .sort("random")
    .limit(n_treatment)
)

control_sample = (
    control_cells.randomColumn(columnName="random", seed=43)
    .sort("random")
    .limit(n_control)
)

treatment_sample = covariates.reduceRegions(
    collection=treatment_sample, reducer=ee.Reducer.mean(), scale=30
)

control_sample = covariates.reduceRegions(
    collection=control_sample, reducer=ee.Reducer.mean(), scale=30
)

all_samples = ee.FeatureCollection([treatment_sample, control_sample]).flatten()

In [ ]:
# Fit a propensity score model
# Each cell gets a score (0-1) indicating its likelihood of being protected

# Convert ee.FeatureCollection to pandas DataFrame
samples_export = all_samples.select(["cell_ID", "elevation", "slope", "protected"])
samples_list = samples_export.getInfo()["features"]
df = pd.DataFrame([feature["properties"] for feature in samples_list])

# Fit logistic regression and predict propensity scores
X = df[["elevation", "slope"]]
y = df["protected"]
model = Pipeline(
    [("scaler", StandardScaler()), ("logit", LogisticRegression(max_iter=1000))]
)
model.fit(X, y)
df["propensity_score"] = model.predict_proba(X)[:, 1]
print(df)

# Examine coefficients to confirm that the data is reproducing the expected PA location bias
predictors = X.columns
logit = model.named_steps["logit"]
coef_df = pd.DataFrame({"variable": predictors, "coefficient": logit.coef_[0]})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df["effect_on_protection"] = coef_df["coefficient"].apply(
    lambda x: "positive" if x > 0 else "negative"
)
print("\nCoefficient summary:")
print(coef_df.sort_values("abs_coefficient", ascending=False))

In [ ]:
# 7. Match each treatment cell to 4 control cells
# Set caliper to 0.2
# Re-using control cells is ok
# Define spatial constraints to enforce similarity between matched cells
# (e.g. within a 50km of PA, within the same ecoregion and country)


# Split treatment and control
treat_df = df[df["protected"] == 1].copy().reset_index(drop=True)
control_df = df[df["protected"] == 0].copy().reset_index(drop=True)

# Convert to 2D array
X_control = control_df[["propensity_score"]].to_numpy()
X_treat = treat_df[["propensity_score"]].to_numpy()

# Fit nearest-neighbor search on control cells
nn = NearestNeighbors(
    n_neighbors=4,
    metric="euclidean",  # in 1D, this is just absolute difference
)
nn.fit(X_control)

# Query the 4 nearest control cells for each treatment cell
distances, indices = nn.kneighbors(X_treat)

# Build a long-form match table
matches = []
caliper = 0.2

for i in range(len(treat_df)):
    treat_row = treat_df.iloc[i]
    treat_id = treat_row["cell_ID"]
    treat_score = treat_row["propensity_score"]

    for rank, (dist, j) in enumerate(zip(distances[i], indices[i]), start=1):
        # Apply caliper
        if dist <= caliper:
            control_row = control_df.iloc[j]

            matches.append(
                {
                    "treat_cell_id": treat_id,
                    "control_cell_id": control_row["cell_ID"],
                    "treat_score": treat_score,
                    "control_score": control_row["propensity_score"],
                    "ps_distance": float(dist),
                    "match_rank": rank,
                }
            )

match_df = pd.DataFrame(matches).sort_values(by="treat_cell_id")

print(match_df)
print(f"\nNumber of matched treatment cells: {match_df['treat_cell_id'].nunique()}")
print(f"Total matched pairs: {len(match_df)}")

# How many matches did each treatment cell get?
match_counts = match_df.groupby("treat_cell_id").size().rename("n_matches")
print(match_counts.value_counts().sort_index())

# Treatment cells with no valid matches inside the caliper
unmatched_treat_ids = set(treat_df["cell_ID"]) - set(match_df["treat_cell_id"])
print(f"Unmatched treatment cells: {len(unmatched_treat_ids)}")

In [ ]:
Map = geemap.Map()

Map.add_basemap("CartoDB.Positron")
# Map.add_basemap("Esri.WorldImagery")

Map.addLayer(
    elevation,
    {"min": 500, "max": 4000, "palette": ["blue", "green", "yellow", "red"]},
    "Elevation",
)
Map.addLayer(
    slope,
    {"min": 0.5, "max": 24.5, "palette": ["blue", "green", "yellow", "red"]},
    "Slope",
)
Map.addLayer(grid_fc, {}, "Grid")
Map.addLayer(treatment_sample, {}, "Treatment")
Map.addLayer(control_sample, {}, "Control")

Map.centerObject(grid_fc)

Map